In [7]:

import os
import time
import random

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
    classification_report
)
# ============================================================
# GOOGLE COLAB SETUP
# ============================================================
# ============================================================
# FEDPROX RESULTS - GOOGLE COLAB
# Flat-FedAvg vs Hier-FedAvg
# ============================================================


# Create folders
os.makedirs("/content/data", exist_ok=True)
os.makedirs("/content/fedavg_results", exist_ok=True)

print("Colab directories ready:")
print("Data    :", "/content/data")
print("Results :", "/content/fedavg_results")



import shutil

shutil.copytree(
    "/content/fedavg_results",
    "/content/drive/MyDrive/fedavg_results",
    dirs_exist_ok=True
)
# ============================================================
# 1. CONFIGURATION & HYPERPARAMETERS
# ============================================================

SEEDS = [42, 101]

NUM_CLIENTS = 20
GLOBAL_ROUNDS = 10
LOCAL_EPOCHS = 3
BATCH_SIZE = 256
LEARNING_RATE = 0.001

TEST_SIZE = 0.20


# ============================================================
# 2. HIERARCHICAL TOPOLOGY
# ============================================================

HFL_CLUSTERS = {
    'Station_1': [0, 1, 2, 3],
    'Station_2': [4, 5, 6, 7],
    'Station_3': [8, 9, 10, 11],
    'Station_4': [12, 13, 14, 15],
    'Station_5': [16, 17, 18, 19]
}


# ============================================================
# 3. DATASET PATHS - GOOGLE COLAB
# ============================================================


file_paths = [
  '/content/drive/MyDrive/MyPaperDatasets/IID_5Clients/client_1_iid.csv',
  '/content/drive/MyDrive/MyPaperDatasets/IID_5Clients/client_2_iid.csv',
  '/content/drive/MyDrive/MyPaperDatasets/IID_5Clients/client_3_iid.csv',
  '/content/drive/MyDrive/MyPaperDatasets/IID_5Clients/client_4_iid.csv',
  '/content/drive/MyDrive/MyPaperDatasets/IID_5Clients/client_5_iid.csv'


]











output_dir = '/content/fedavg_results'

os.makedirs(output_dir, exist_ok=True)


# ============================================================
# 4. LABEL MAPPING
# ============================================================


label_mapping = {
    0: 0,
    1: 1,
    2: 2,
    3: 3,
    4: 4,
    9: 5
}


# ============================================================
# 5. LOAD CLIENT DATA
# ============================================================

def load_client_data(paths):

    client_dfs = []

    for client_id, path in enumerate(paths):

        if not os.path.exists(path):

            raise FileNotFoundError(
                f"File does not exist:\n{path}"
            )

        df = pd.read_csv(path)

        if df.empty:

            raise ValueError(
                f"Client {client_id + 1} dataset is empty."
            )

        if 'label' not in df.columns:

            raise ValueError(
                f"'label' column missing in "
                f"Client {client_id + 1}"
            )

        client_dfs.append(df)

    return client_dfs


client_dfs = load_client_data(file_paths)
# ============================================================
# 6. COMBINE DATA FOR GLOBAL TRAIN/TEST SPLIT
# ============================================================

combined_df = pd.concat(
    client_dfs,
    ignore_index=True
)

combined_df['label'] = (
    combined_df['label']
    .map(label_mapping)
)

if combined_df['label'].isna().any():

    raise ValueError(
        "Unknown labels detected after label mapping."
    )


X_full_raw = (
    combined_df
    .drop('label', axis=1)
    .values
    .astype(np.float32)
)

y_full = (
    combined_df['label']
    .values
    .astype(np.int32)
)


X_full_raw = np.nan_to_num(
    X_full_raw,
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)


# ============================================================
# 7. INDEPENDENT TRAIN / TEST SPLIT
# ============================================================

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_full_raw,
    y_full,
    test_size=TEST_SIZE,
    random_state=42,
    stratify=y_full
)


# ============================================================
# 8. STANDARDIZATION
# ============================================================

scaler = StandardScaler()

X_train = scaler.fit_transform(
    X_train_raw
).astype(np.float32)

X_test = scaler.transform(
    X_test_raw
).astype(np.float32)

input_shape = (
    X_train.shape[1],
)

num_classes = len(
    np.unique(y_full)
)


# ============================================================
# 9. CREATE CLIENT DATA
# ============================================================

client_data_list = []

for client_id, df in enumerate(client_dfs):

    df = df.copy()

    df['label'] = (
        df['label']
        .map(label_mapping)
    )

    if df['label'].isna().any():

        raise ValueError(
            f"Unknown labels in Client {client_id + 1}"
        )

    X_c_raw = (
        df
        .drop('label', axis=1)
        .values
        .astype(np.float32)
    )

    y_c = (
        df['label']
        .values
        .astype(np.int32)
    )

    X_c_raw = np.nan_to_num(
        X_c_raw,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    X_c = scaler.transform(
        X_c_raw
    ).astype(np.float32)

    client_data_list.append(
        (X_c, y_c)
    )


# ============================================================
# 10. PRINT DATA INFORMATION
# ============================================================

print("\n" + "=" * 75)
print("DATASET INFORMATION")
print("=" * 75)

print(
    f"Total samples      : {len(X_full_raw)}"
)

print(
    f"Training samples   : {len(X_train)}"
)

print(
    f"Testing samples    : {len(X_test)}"
)

print(
    f"Number of features : {input_shape[0]}"
)

print(
    f"Number of classes  : {num_classes}"
)

print("\nClient distribution:")

for client_idx, (X_c, y_c) in enumerate(
    client_data_list
):

    unique, counts = np.unique(
        y_c,
        return_counts=True
    )

    distribution = {
        int(k): int(v)
        for k, v in zip(
            unique,
            counts
        )
    }

    print(
        f"Client {client_idx + 1}: "
        f"{len(X_c)} samples | "
        f"{distribution}"
    )


print("\nTest-set distribution:")

unique_test, counts_test = np.unique(
    y_test,
    return_counts=True
)

for cls, count in zip(
    unique_test,
    counts_test
):

    print(
        f"Class {cls}: {count}"
    )


# ============================================================
# 11. MODEL ARCHITECTURE
# ============================================================

def create_mlp_model(
    input_shape,
    num_classes
):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=input_shape
        ),

        tf.keras.layers.Dense(
            128,
            activation='relu'
        ),

        tf.keras.layers.Dense(
            64,
            activation='relu'
        ),

        tf.keras.layers.Dense(
            32,
            activation='relu'
        ),

        tf.keras.layers.Dense(
            num_classes,
            activation='softmax'
        )
    ])

    return model


# ============================================================
# 12. MODEL SIZE
# ============================================================

def calculate_model_size_mb(model):

    total_params = model.count_params()

    return (
        total_params * 4
    ) / (
        1024 * 1024
    )


MODEL_SIZE_MB = calculate_model_size_mb(
    create_mlp_model(
        input_shape,
        num_classes
    )
)


print("\n" + "=" * 75)
print(
    f"MODEL SIZE: {MODEL_SIZE_MB:.4f} MB"
)
print("=" * 75)


# ============================================================
# 13. WEIGHT AGGREGATION
# ============================================================

def aggregate_weights(
    weights_list,
    sizes_list
):

    total_samples = sum(
        sizes_list
    )

    aggregated_weights = []

    for layer_idx in range(
        len(weights_list[0])
    ):

        layer_avg = np.zeros_like(
            weights_list[0][layer_idx]
        )

        for client_idx in range(
            len(weights_list)
        ):

            factor = (
                sizes_list[client_idx]
                / total_samples
            )

            layer_avg += (
                factor
                * weights_list[client_idx][layer_idx]
            )

        aggregated_weights.append(
            layer_avg
        )

    return aggregated_weights


# ============================================================
# 14. FEDAVG CLIENT TRAINING
# ============================================================

def train_client_fedavg(
    global_weights,
    X_c,
    y_c
):

    model = create_mlp_model(
        input_shape,
        num_classes
    )

    model.set_weights(
        global_weights
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),
        loss='sparse_categorical_crossentropy'
    )

    model.fit(
        X_c,
        y_c,
        epochs=LOCAL_EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=0
    )

    return model.get_weights()


# ============================================================
# 15. EVALUATION FUNCTION
# ============================================================

def evaluate_model(model):

    probabilities = model.predict(
        X_test,
        batch_size=BATCH_SIZE,
        verbose=0
    )

    predictions = np.argmax(
        probabilities,
        axis=1
    )

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    macro_precision = precision_score(
        y_test,
        predictions,
        average='macro',
        zero_division=0
    )

    macro_recall = recall_score(
        y_test,
        predictions,
        average='macro',
        zero_division=0
    )

    macro_f1 = f1_score(
        y_test,
        predictions,
        average='macro',
        zero_division=0
    )

    class_report = classification_report(
        y_test,
        predictions,
        labels=np.arange(num_classes),
        output_dict=True,
        zero_division=0
    )

    cm = confusion_matrix(
        y_test,
        predictions,
        labels=np.arange(num_classes)
    )

    # --------------------------------------------------------
    # Safe normalized confusion matrix
    # --------------------------------------------------------

    row_sums = cm.sum(
        axis=1,
        keepdims=True
    )

    cm_norm = np.divide(
        cm.astype(np.float64),
        row_sums,
        out=np.zeros_like(
            cm,
            dtype=np.float64
        ),
        where=row_sums != 0
    )

    return {

        'accuracy':
            accuracy,

        'macro_precision':
            macro_precision,

        'macro_recall':
            macro_recall,

        'macro_f1':
            macro_f1,

        'class_report':
            class_report,

        'cm_norm':
            cm_norm
    }


# ============================================================
# 16. FL EXPERIMENT
# ============================================================

def run_fl_experiment(
    method,
    seed
):

    # --------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------

    np.random.seed(seed)
    random.seed(seed)
    tf.random.set_seed(seed)

    # --------------------------------------------------------
    # Global model
    # --------------------------------------------------------

    global_model = create_mlp_model(
        input_shape,
        num_classes
    )

    total_comm_mb = 0.0

    experiment_start = time.time()

    round_accuracy = []
    round_f1 = []
    round_precision = []
    round_recall = []
    round_time = []

    # ========================================================
    # GLOBAL ROUNDS
    # ========================================================

    for round_idx in range(
        GLOBAL_ROUNDS
    ):

        round_start = time.time()

        global_weights = (
            global_model.get_weights()
        )

        # ====================================================
        # FLAT FEDAVG
        # ====================================================

        if method == 'Flat-FedAvg':

            local_weights = []
            local_sizes = []

            for client_idx in range(
                NUM_CLIENTS
            ):

                X_c, y_c = (
                    client_data_list[
                        client_idx
                    ]
                )

                client_weights = (
                    train_client_fedavg(
                        global_weights,
                        X_c,
                        y_c
                    )
                )

                local_weights.append(
                    client_weights
                )

                local_sizes.append(
                    len(X_c)
                )

                # Client -> Global Server
                # Global Server -> Client
                total_comm_mb += (
                    MODEL_SIZE_MB * 2
                )

            # ------------------------------------------------
            # Global aggregation
            # ------------------------------------------------

            new_global_weights = (
                aggregate_weights(
                    local_weights,
                    local_sizes
                )
            )

            global_model.set_weights(
                new_global_weights
            )

        # ====================================================
        # HIERARCHICAL FEDAVG
        # ====================================================

        elif method == 'Hier-FedAvg':

            station_weights = []
            station_sizes = []

            for station, client_indices in (
                HFL_CLUSTERS.items()
            ):

                station_local_weights = []
                station_local_sizes = []

                # --------------------------------------------
                # Client training
                # --------------------------------------------

                for client_idx in client_indices:

                    X_c, y_c = (
                        client_data_list[
                            client_idx
                        ]
                    )

                    client_weights = (
                        train_client_fedavg(
                            global_weights,
                            X_c,
                            y_c
                        )
                    )

                    station_local_weights.append(
                        client_weights
                    )

                    station_local_sizes.append(
                        len(X_c)
                    )

                    # Client <-> Station
                    total_comm_mb += (
                        MODEL_SIZE_MB * 2
                    )

                # --------------------------------------------
                # Station aggregation
                # --------------------------------------------

                station_agg_weights = (
                    aggregate_weights(
                        station_local_weights,
                        station_local_sizes
                    )
                )

                station_weights.append(
                    station_agg_weights
                )

                station_sizes.append(
                    sum(station_local_sizes)
                )

                # --------------------------------------------
                # Station <-> Global Server
                # --------------------------------------------

                total_comm_mb += (
                    MODEL_SIZE_MB * 2
                )

            # -----------------------------------------------
            # Global aggregation
            # -----------------------------------------------

            new_global_weights = (
                aggregate_weights(
                    station_weights,
                    station_sizes
                )
            )

            global_model.set_weights(
                new_global_weights
            )

        else:

            raise ValueError(
                f"Unknown FL method: {method}"
            )

        # ====================================================
        # ROUND EVALUATION
        # ====================================================

        round_result = evaluate_model(
            global_model
        )

        round_accuracy.append(
            round_result['accuracy']
        )

        round_f1.append(
            round_result['macro_f1']
        )

        round_precision.append(
            round_result['macro_precision']
        )

        round_recall.append(
            round_result['macro_recall']
        )

        elapsed_round = (
            time.time()
            - round_start
        )

        round_time.append(
            elapsed_round
        )

        print(
            f"      Round {round_idx + 1:02d} | "
            f"Accuracy: "
            f"{round_result['accuracy']:.4f} | "
            f"Macro-F1: "
            f"{round_result['macro_f1']:.4f}"
        )

    # ========================================================
    # FINAL EVALUATION
    # ========================================================

    final_result = evaluate_model(
        global_model
    )

    total_time = (
        time.time()
        - experiment_start
    )

    return {

        'accuracy':
            final_result['accuracy'],

        'macro_precision':
            final_result['macro_precision'],

        'macro_recall':
            final_result['macro_recall'],

        'macro_f1':
            final_result['macro_f1'],

        'comm_mb':
            total_comm_mb,

        'time_sec':
            total_time,

        'class_report':
            final_result['class_report'],

        'cm_norm':
            final_result['cm_norm'],

        'round_accuracy':
            round_accuracy,

        'round_f1':
            round_f1,

        'round_precision':
            round_precision,

        'round_recall':
            round_recall,

        'round_time':
            round_time
    }


# ============================================================
# 17. METHODS
# ============================================================

methods = [
    'Flat-FedAvg',
    'Hier-FedAvg'
]


# ============================================================
# 18. RUN BENCHMARK
# ============================================================

summary_results = []

detailed_class_metrics = {
    method: []
    for method in methods
}

convergence_results = []


print("\n" + "=" * 75)
print("MULTI-SEED FEDAVG BENCHMARK")
print("=" * 75)

print(
    "Flat-FedAvg vs Hier-FedAvg"
)

print("=" * 75)


for method in methods:

    print("\n" + "-" * 75)

    print(
        f"METHOD: {method}"
    )

    print("-" * 75)

    accuracy_values = []
    f1_values = []
    precision_values = []
    recall_values = []
    communication_values = []
    time_values = []

    for seed in SEEDS:

        print(
            f"\n  Seed {seed}"
        )

        result = run_fl_experiment(
            method,
            seed
        )

        accuracy_values.append(
            result['accuracy']
        )

        f1_values.append(
            result['macro_f1']
        )

        precision_values.append(
            result['macro_precision']
        )

        recall_values.append(
            result['macro_recall']
        )

        communication_values.append(
            result['comm_mb']
        )

        time_values.append(
            result['time_sec']
        )

        detailed_class_metrics[
            method
        ].append(
            result
        )

        # ----------------------------------------------------
        # Save convergence
        # ----------------------------------------------------

        for round_idx in range(
            GLOBAL_ROUNDS
        ):

            convergence_results.append({

                'Method':
                    method,

                'Seed':
                    seed,

                'Round':
                    round_idx + 1,

                'Accuracy':
                    result[
                        'round_accuracy'
                    ][round_idx],

                'Macro_F1':
                    result[
                        'round_f1'
                    ][round_idx],

                'Macro_Precision':
                    result[
                        'round_precision'
                    ][round_idx],

                'Macro_Recall':
                    result[
                        'round_recall'
                    ][round_idx],

                'Round_Time_Sec':
                    result[
                        'round_time'
                    ][round_idx]
            })

        print(
            f"  Final Accuracy: "
            f"{result['accuracy']:.4f}"
        )

        print(
            f"  Final Macro-F1: "
            f"{result['macro_f1']:.4f}"
        )

        print(
            f"  Communication: "
            f"{result['comm_mb']:.2f} MB"
        )

        print(
            f"  Time: "
            f"{result['time_sec']:.2f} sec"
        )

    # ========================================================
    # SUMMARY
    # ========================================================

    summary_results.append({

        'Method':
            method,

        'Accuracy_Mean':
            np.mean(
                accuracy_values
            ),

        'Accuracy_Std':
            np.std(
                accuracy_values
            ),

        'Macro_F1_Mean':
            np.mean(
                f1_values
            ),

        'Macro_F1_Std':
            np.std(
                f1_values
            ),

        'Macro_Precision_Mean':
            np.mean(
                precision_values
            ),

        'Macro_Precision_Std':
            np.std(
                precision_values
            ),

        'Macro_Recall_Mean':
            np.mean(
                recall_values
            ),

        'Macro_Recall_Std':
            np.std(
                recall_values
            ),

        'Comm_Cost_MB_Mean':
            np.mean(
                communication_values
            ),

        'Comm_Cost_MB_Std':
            np.std(
                communication_values
            ),

        'Exec_Time_Sec_Mean':
            np.mean(
                time_values
            ),

        'Exec_Time_Sec_Std':
            np.std(
                time_values
            )
    })


# ============================================================
# 19. OVERALL SUMMARY
# ============================================================

df_summary = pd.DataFrame(
    summary_results
)

summary_path = os.path.join(
    output_dir,
    'fedavg_summary.csv'
)

df_summary.to_csv(
    summary_path,
    index=False
)


print("\n" + "=" * 75)
print("OVERALL FEDAVG BENCHMARK SUMMARY")
print("=" * 75)

print(
    "Mean ± Std across seeds"
)

print("=" * 75)


for _, row in df_summary.iterrows():

    print(
        f"\n{row['Method']}"
    )

    print(
        f"Accuracy: "
        f"{row['Accuracy_Mean']:.4f} ± "
        f"{row['Accuracy_Std']:.4f}"
    )

    print(
        f"Macro Precision: "
        f"{row['Macro_Precision_Mean']:.4f} ± "
        f"{row['Macro_Precision_Std']:.4f}"
    )

    print(
        f"Macro Recall: "
        f"{row['Macro_Recall_Mean']:.4f} ± "
        f"{row['Macro_Recall_Std']:.4f}"
    )

    print(
        f"Macro F1: "
        f"{row['Macro_F1_Mean']:.4f} ± "
        f"{row['Macro_F1_Std']:.4f}"
    )

    print(
        f"Communication: "
        f"{row['Comm_Cost_MB_Mean']:.2f} ± "
        f"{row['Comm_Cost_MB_Std']:.2f} MB"
    )

    print(
        f"Execution Time: "
        f"{row['Exec_Time_Sec_Mean']:.2f} ± "
        f"{row['Exec_Time_Sec_Std']:.2f} sec"
    )


# ============================================================
# 20. CONVERGENCE CSV
# ============================================================

df_convergence = pd.DataFrame(
    convergence_results
)

convergence_path = os.path.join(
    output_dir,
    'fedavg_convergence_results.csv'
)

df_convergence.to_csv(
    convergence_path,
    index=False
)


# ============================================================
# 21. CLASS-WISE PERFORMANCE
# ============================================================

print("\n" + "=" * 75)
print("FEDAVG CLASS-WISE PERFORMANCE")
print("=" * 75)


for method in methods:

    reports = [
        run['class_report']
        for run in detailed_class_metrics[
            method
        ]
    ]

    class_rows = []

    for c in range(
        num_classes
    ):

        c_str = str(c)

        precision_values = [
            report[c_str]['precision']
            for report in reports
        ]

        recall_values = [
            report[c_str]['recall']
            for report in reports
        ]

        f1_values_class = [
            report[c_str]['f1-score']
            for report in reports
        ]

        support_values = [
            report[c_str]['support']
            for report in reports
        ]

        class_rows.append({

            'Class':
                f'Class {c}',

            'Support':
                np.mean(
                    support_values
                ),

            'Precision_Mean':
                np.mean(
                    precision_values
                ),

            'Precision_Std':
                np.std(
                    precision_values
                ),

            'Recall_Mean':
                np.mean(
                    recall_values
                ),

            'Recall_Std':
                np.std(
                    recall_values
                ),

            'F1_Mean':
                np.mean(
                    f1_values_class
                ),

            'F1_Std':
                np.std(
                    f1_values_class
                )
        })

    df_class = pd.DataFrame(
        class_rows
    )

    print(
        f"\n--- {method} ---"
    )

    print(
        df_class.round(4).to_string(
            index=False
        )
    )

    filename = (
        method.lower()
        .replace('-', '_')
        + '_class_wise_performance.csv'
    )

    df_class.to_csv(
        os.path.join(
            output_dir,
            filename
        ),
        index=False
    )


# ============================================================
# 22. FEDAVG CONFUSION MATRICES
#     CSV + HIGH-RESOLUTION PNG
# ============================================================

print("\n" + "=" * 75)
print("FEDAVG - AVERAGE NORMALIZED CONFUSION MATRICES")
print("=" * 75)


for method in methods:

    print("\n" + "-" * 75)
    print(
        f"CONFUSION MATRIX: {method}"
    )
    print("-" * 75)

    # --------------------------------------------------------
    # Collect confusion matrices from all seeds
    # --------------------------------------------------------

    matrices = [
        run['cm_norm']
        for run in detailed_class_metrics[
            method
        ]
    ]

    # --------------------------------------------------------
    # Average across seeds
    # --------------------------------------------------------

    avg_cm = np.mean(
        matrices,
        axis=0
    )

    # --------------------------------------------------------
    # Create DataFrame
    # --------------------------------------------------------

    df_cm = pd.DataFrame(
        avg_cm,
        index=[
            f'Class {i}'
            for i in range(num_classes)
        ],
        columns=[
            f'Class {i}'
            for i in range(num_classes)
        ]
    )

    # --------------------------------------------------------
    # Filename base
    # --------------------------------------------------------

    filename_base = (
        method
        .lower()
        .replace('-', '_')
    )

    # ========================================================
    # SAVE CSV
    # ========================================================

    csv_path = os.path.join(
        output_dir,
        filename_base +
        '_normalized_confusion_matrix.csv'
    )

    df_cm.to_csv(
        csv_path
    )

    # --------------------------------------------------------
    # Print matrix
    # --------------------------------------------------------

    print(
        "\nAverage Normalized Confusion Matrix:"
    )

    print(
        df_cm.round(4).to_string()
    )

    # ========================================================
    # CREATE PNG
    # ========================================================

    fig, ax = plt.subplots(
        figsize=(9, 8)
    )

    im = ax.imshow(
        avg_cm,
        interpolation='nearest'
    )

    # --------------------------------------------------------
    # Color bar
    # --------------------------------------------------------

    cbar = ax.figure.colorbar(
        im,
        ax=ax
    )

    cbar.ax.set_ylabel(
        'Normalized Value',
        rotation=-90,
        va='bottom'
    )

    # --------------------------------------------------------
    # Axis labels
    # --------------------------------------------------------

    ax.set(
        xticks=np.arange(num_classes),
        yticks=np.arange(num_classes),

        xticklabels=[
            f'Class {i}'
            for i in range(num_classes)
        ],

        yticklabels=[
            f'Class {i}'
            for i in range(num_classes)
        ],

        ylabel='True Label',
        xlabel='Predicted Label',

        title=(
            f'{method}\n'
            'Average Normalized Confusion Matrix'
        )
    )

    # --------------------------------------------------------
    # Rotate x-axis labels
    # --------------------------------------------------------

    plt.setp(
        ax.get_xticklabels(),
        rotation=45,
        ha='right',
        rotation_mode='anchor'
    )

    # --------------------------------------------------------
    # Write values inside cells
    # --------------------------------------------------------

    threshold = avg_cm.max() / 2.0

    for i in range(num_classes):

        for j in range(num_classes):

            value = avg_cm[i, j]

            ax.text(
                j,
                i,
                f'{value:.2f}',
                ha='center',
                va='center',
                color=(
                    'white'
                    if value > threshold
                    else 'black'
                ),
                fontsize=11
            )

    # --------------------------------------------------------
    # Layout
    # --------------------------------------------------------

    fig.tight_layout()

    # ========================================================
    # SAVE PNG - 300 DPI
    # ========================================================

    image_path = os.path.join(
        output_dir,
        filename_base +
        '_normalized_confusion_matrix.png'
    )

    plt.savefig(
        image_path,
        dpi=300,
        bbox_inches='tight'
    )

    plt.close(fig)

    # --------------------------------------------------------
    # Print paths
    # --------------------------------------------------------

    print("\nCSV saved to:")
    print(csv_path)

    print("\nPNG saved to:")
    print(image_path)


# ============================================================
# 23. FINAL OUTPUT
# ============================================================

print("\n" + "=" * 75)
print("[FINISHED] FEDAVG BENCHMARK COMPLETED")
print("=" * 75)

print("\nMethods evaluated:")

for method in methods:

    print(
        f"  ✓ {method}"
    )


print("\nMetrics generated:")

print("  ✓ Accuracy")
print("  ✓ Macro Precision")
print("  ✓ Macro Recall")
print("  ✓ Macro F1")
print("  ✓ Class-wise Precision")
print("  ✓ Class-wise Recall")
print("  ✓ Class-wise F1")
print("  ✓ Class Support")
print("  ✓ Normalized Confusion Matrices")
print("  ✓ Confusion Matrix PNG Images")
print("  ✓ Round-by-round Convergence")
print("  ✓ Communication Cost")
print("  ✓ Training/Execution Time")
print("  ✓ Mean ± Std across seeds")


print("\nSummary saved to:")
print(summary_path)

print("\nConvergence saved to:")
print(convergence_path)

print("\nAll results saved in:")
print(output_dir)

print("\nGenerated confusion matrix files:")

for method in methods:

    filename_base = (
        method
        .lower()
        .replace('-', '_')
    )

    print(
        f"  ✓ {filename_base}_normalized_confusion_matrix.csv"
    )

    print(
        f"  ✓ {filename_base}_normalized_confusion_matrix.png"
    )

print("\n" + "=" * 75)


Colab directories ready:
Data    : /content/data
Results : /content/fedavg_results


FileNotFoundError: File does not exist:
/content/drive/MyDrive/MyPaperDatasets/IID_5Clients/client_1_iid.csv

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
